In [ ]:
import pandas as pd
import xarray as xr
from pathlib import Path
import tempfile

from oggm import cfg, utils, workflow, tasks, global_tasks

import gungnir

In [ ]:
region = "11"
rgi_id = "RGI60-11.03646"
years_range = [2020, 2021]

In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir_str:
    working_dir = Path(tmp_dir_str)

base_url = 'https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/elev_bands/'

print("Working directory:", working_dir)

cfg.initialize()

# Settings
cfg.PATHS['working_dir'] = working_dir
cfg.PARAMS['use_multiprocessing'] = True # use all available CPUs
cfg.PARAMS['border'] = 10
cfg.PARAMS['hydro_month_nh'] = 1
cfg.PARAMS['dl_verify'] = False
cfg.PARAMS['continue_on_error'] = True

# Now we initialize the glacier directories
gdirs = workflow.init_glacier_directories([rgi_id],
                                            prepro_base_url=base_url,
                                            from_prepro_level=2)

# We execute the entity tasks
list_tasks = [tasks.gridded_attributes, tasks.glacier_masks]

for task in list_tasks:
    workflow.execute_entity_task(task, gdirs)


In [ ]:
cache_path = Path(gungnir.era5_climate.gungnir_path) / ".cache"
cache_sample_path = Path(gungnir.era5_climate.gungnir_path) / ".cache_sample" / "ERA5"
cache_sample_path.mkdir(parents=True, exist_ok=True)
monthly_nc = Path(cache_path) / "ERA5" / f"era5_land_monthly_region_{region}.nc"
geopotential_nc = Path(cache_path) / "ERA5" / f"era5_land_geopotential_region_{region}.nc"
assert monthly_nc.exists()
assert geopotential_nc.exists()
monthly_ds = xr.open_dataset(monthly_nc)
geopotential_ds = xr.open_dataset(geopotential_nc)

## Monthly data

In [ ]:
monthly_ds

In [ ]:
gdir = gdirs[0]
lat = float(gdir.cenlat)
lon = float(gdir.cenlon)
print(f"{lat=}")
print(f"{lon=}")
lats = monthly_ds.latitude
lons = monthly_ds.longitude

lat0 = lats.values[0]
lat1 = lats.values[-1]
lon0 = lons.values[0]
lon1 = lons.values[-1]
lat_lower = lats.where(lats <= lat, drop=True).max().item()
lat_upper = lats.where(lats >= lat, drop=True).min().item()
lon_lower = lons.where(lons <= lon, drop=True).max().item()
lon_upper = lons.where(lons >= lon, drop=True).min().item()

if lat0 < lat1:  # increasing
    if lon0 < lon1:  # increasing
        ds_subset = monthly_ds.sel(latitude=slice(lat_lower, lat_upper), longitude=slice(lon_lower, lon_upper), time=slice(f"{years_range[0]}-01-01", f"{years_range[1]}-12-31"))
    else:            # decreasing
        ds_subset = monthly_ds.sel(latitude=slice(lat_lower, lat_upper), longitude=slice(lon_upper, lon_lower), time=slice(f"{years_range[0]}-01-01", f"{years_range[1]}-12-31"))
else:            # decreasing
    if lon0 < lon1:  # increasing
        ds_subset = monthly_ds.sel(latitude=slice(lat_upper, lat_lower), longitude=slice(lon_lower, lon_upper), time=slice(f"{years_range[0]}-01-01", f"{years_range[1]}-12-31"))
    else:            # decreasing
        ds_subset = monthly_ds.sel(latitude=slice(lat_upper, lat_lower), longitude=slice(lon_upper, lon_lower), time=slice(f"{years_range[0]}-01-01", f"{years_range[1]}-12-31"))
print(ds_subset)

In [ ]:
# Write to NetCDF with compression
encoding = {name: {"zlib": True, "complevel": 4} for name in ds_subset.data_vars}
ds_subset.to_netcdf(cache_sample_path / f"era5_land_monthly_region_{region}.nc", encoding=encoding)

## Geopotential

In [ ]:
geopotential_ds

In [ ]:
geopotential_ds.longitude

In [ ]:
gdir = gdirs[0]
lat = float(gdir.cenlat)
lon = float(gdir.cenlon)
print(f"{lat=}")
print(f"{lon=}")

# Correct non monotonous longitude
if (geopotential_ds.longitude>180).any():
    print("Correcting")
    geopotential_ds = geopotential_ds.assign_coords(
        longitude=((geopotential_ds.longitude + 180) % 360) - 180
    ).sortby("longitude")

lats = geopotential_ds.latitude
lons = geopotential_ds.longitude

lat0 = lats.values[0]
lat1 = lats.values[-1]
lon0 = lons.values[0]
lon1 = lons.values[-1]
lat_lower = lats.where(lats <= lat, drop=True).max().item()
lat_upper = lats.where(lats >= lat, drop=True).min().item()
lon_lower = lons.where(lons <= lon, drop=True).max().item()
lon_upper = lons.where(lons >= lon, drop=True).min().item()

if lat0 < lat1:  # increasing
    if lon0 < lon1:  # increasing
        ds_subset = geopotential_ds.sel(latitude=slice(lat_lower, lat_upper), longitude=slice(lon_lower, lon_upper))
    else:            # decreasing
        ds_subset = geopotential_ds.sel(latitude=slice(lat_lower, lat_upper), longitude=slice(lon_upper, lon_lower))
else:            # decreasing
    if lon0 < lon1:  # increasing
        ds_subset = geopotential_ds.sel(latitude=slice(lat_upper, lat_lower), longitude=slice(lon_lower, lon_upper))
    else:            # decreasing
        ds_subset = geopotential_ds.sel(latitude=slice(lat_upper, lat_lower), longitude=slice(lon_upper, lon_lower))
print(ds_subset)

In [ ]:
# Write to NetCDF with compression
encoding = {name: {"zlib": True, "complevel": 4} for name in ds_subset.data_vars}
ds_subset.to_netcdf(cache_sample_path / f"era5_land_geopotential_region_{region}.nc", encoding=encoding)